# BYV-AI Test Evaluation: BYV Markup v0.1 + Qwen3-4B-Instruct-2507

**Base model:** `Qwen/Qwen3-4B-Instruct-2507` (pinned revision `cdbee75f17c01a7cc42f958dc650907174af0554`)  
**Dataset:** BYV Markup v0.1, held-out test only (35 records)  
**Adapter:** `byv-ai-qwen3-4b-instruct-2507-byv-markup-inline-context-qlora-t4-v0.2`  
**Purpose:** measure output-mode accuracy, BYV syntax validity, and exact tool-call agreement.

Notebook ini hanya membaca `test.jsonl`. Ia tidak membaca train/eval split dan tidak mengeksekusi tool. Instruksi BYV dan schema tool dibangun inline dengan fungsi yang sama seperti training, tanpa mengunduh system-prompt template. Syntax atau exact-match bukan semantic accuracy; jawaban TEXT tetap memerlukan review manusia.


In [3]:
%pip install -q -U "transformers>=4.51.0,<5.0.0" "accelerate>=1.0.0,<2.0.0" "datasets>=3.0.0,<5.0.0" "peft>=0.14.0,<1.0.0" "bitsandbytes>=0.45.0,<1.0.0" "safetensors>=0.4.5" "packaging>=23.0"


In [4]:
import os, sys, json, re, hashlib, inspect, platform
from pathlib import Path
from collections import Counter
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from packaging.version import Version

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if Version(torch.__version__.split('+')[0]) < Version("2.6.0"):
    raise RuntimeError("Qwen3-2507 recommends PyTorch >= 2.6.0.")
if not torch.cuda.is_available():
    raise RuntimeError("Evaluation requires a CUDA GPU for this 4B QLoRA setup.")


Python: 3.13.16
PyTorch: 2.11.0+cu130
Transformers: 4.57.6
CUDA available: True


In [5]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_REVISION = "cdbee75f17c01a7cc42f958dc650907174af0554"
DATASET_VERSION = "byv-markup-v0.1"
ZIP_NAME = "BYV-AI_dataset_BYV-Markup_v0.1.zip"
ZIP_ROOT = "BYV-AI_dataset_byv_markup_v0.1/"
EXPECTED_TEST_SHA256 = "518fc6bdaee0cf8e48f2b070898a6f20e35bdff0d6b2202c6a4699fd2755b952"
RUN_NAME = "byv-ai-qwen3-4b-instruct-2507-byv-markup-inline-context-qlora-t4-v0.2"
MAX_NEW_TOKENS = 1024
PROMPT_STRATEGY = "inline-byv-instructions-plus-per-record-tool-schema-v1"
DATASET_SYSTEM_SENTINEL = "__BYV_SYSTEM_PROMPT_TEMPLATE_INJECTED_AT_RUNTIME__"

from google.colab import drive, files
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/BYV-AI')
ARCHIVE_PATH = DRIVE_ROOT / 'datasets' / ZIP_NAME
OUTPUT_DIR = DRIVE_ROOT / 'outputs' / RUN_NAME
EXPERIMENT_DIR = DRIVE_ROOT / 'experiments' / RUN_NAME
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
if not ARCHIVE_PATH.exists():
    print(f"Upload {ZIP_NAME} once. It will be saved to Google Drive.")
    uploaded = files.upload()
    if ZIP_NAME not in uploaded:
        raise FileNotFoundError(f"Please upload the exact file: {ZIP_NAME}")
    ARCHIVE_PATH.parent.mkdir(parents=True, exist_ok=True)
    ARCHIVE_PATH.write_bytes(uploaded[ZIP_NAME])
    del uploaded
if not (OUTPUT_DIR / "adapter_config.json").exists():
    raise FileNotFoundError(f"LoRA adapter not found: {OUTPUT_DIR}. Run the training notebook first.")
print("Dataset archive:", ARCHIVE_PATH)
print("Adapter:", OUTPUT_DIR)


Mounted at /content/drive
Dataset archive: /content/drive/MyDrive/BYV-AI/datasets/BYV-AI_dataset_BYV-Markup_v0.1.zip
Adapter: /content/drive/MyDrive/BYV-AI/outputs/byv-ai-qwen3-4b-instruct-2507-byv-markup-inline-context-qlora-t4-v0.2


## 1. Build inline BYV context and load the held-out test split only

In [6]:
BASE_SYSTEM = """You are BYV-AI, a helpful text assistant.
Follow the user's instructions carefully. Respond in the user's language. Use plain text by default, Markdown when useful, and complete accurate source code when requested.
BYV markup is ordinary assistant text, not native tool calling. Do not emit BYV when no available tool is needed. Never invent tools, parameters, or tool results. Never claim an action succeeded without an actual gateway result.
After actual tool results are provided, answer naturally in plain text or Markdown unless another tool action is required.
"""

BYV_TOOL_INSTRUCTIONS = """When a requested action requires an available tool, output exactly one BYV calls block with one invoke per requested tool.
Use this structure and attribute order exactly:
<|BYV| calls>
<|BYV| invoke name="tool.name" id="call_001">
<|BYV| parameter name="parameter_name" type="string">raw value</|BYV| parameter>
</|BYV| invoke>
</|BYV| calls>
String parameters use type="string" and raw UTF-8 without JSON quotes. Escape backslash as \\\\, newline as \\n, carriage return as \\r, tab as \\t, and a literal closing parameter token by prefixing it with a backslash.
Number, boolean, null, array, and object parameters use the matching BYV type and exactly one valid JSON value. Schema integers use type="number" and an integer value.
Use exact tool and parameter names from the supplied schemas. Do not invent tools, parameters, or results. Do not put prose or Markdown inside a calls block.
Use STRUCTURED for tool-only requests, MIXED only when useful explanatory text belongs outside the block, and TEXT when no tool call is needed.
Never claim a tool ran or succeeded before the gateway returns its actual result.
"""

def build_system_prompt(tools):
    if not tools:
        return BASE_SYSTEM + "\nNo tools are available for this request. Answer in TEXT mode and do not emit BYV calls."
    tool_schema_text = json.dumps(tools, ensure_ascii=False, separators=(",", ":"))
    return (
        BASE_SYSTEM
        + "\n\n"
        + BYV_TOOL_INSTRUCTIONS
        + "\n\nAVAILABLE TOOLS AND EXACT SCHEMAS:\n"
        + tool_schema_text
    )

def messages_with_inline_context(row):
    messages = [dict(message) for message in row["messages"]]
    if (not messages or messages[0].get("role") != "system"
            or messages[0].get("content") != DATASET_SYSTEM_SENTINEL):
        raise ValueError(f"{row.get('id')}: dataset system placeholder is missing or invalid")
    messages[0]["content"] = build_system_prompt(row.get("tools", []))
    return messages


import zipfile
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    member = ZIP_ROOT + "test.jsonl"
    if member not in archive.namelist():
        raise FileNotFoundError(member)
    test_payload = archive.read(member)  # held-out test only; train/eval are not opened
actual_test_hash = hashlib.sha256(test_payload).hexdigest()
if actual_test_hash != EXPECTED_TEST_SHA256:
    raise RuntimeError(f"Held-out test SHA-256 mismatch: {actual_test_hash}")
TEST_PATH = EXPERIMENT_DIR / "test.jsonl"
TEST_PATH.write_bytes(test_payload)

def load_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]
TEST_ROWS = load_jsonl(TEST_PATH)
print("Held-out test records:", len(TEST_ROWS))
print("Inline BYV prompt strategy:", PROMPT_STRATEGY)


Held-out test records: 35
Inline BYV prompt strategy: inline-byv-instructions-plus-per-record-tool-schema-v1


## 2. BYV parser for evaluation

Parser ini hanya dipakai untuk mengukur output model. Ia tidak mengeksekusi tool.

In [7]:
CALLS_OPEN = "<|BYV| calls>"
CALLS_CLOSE = "</|BYV| calls>"
INVOKE_CLOSE = "</|BYV| invoke>"
PARAM_CLOSE = "</|BYV| parameter>"
INVOKE_OPEN_RE = re.compile(r'<\|BYV\| invoke name="([A-Za-z_][A-Za-z0-9_.:-]*)" id="([A-Za-z0-9][A-Za-z0-9._:-]{0,63})">')
PARAM_OPEN_RE = re.compile(r'<\|BYV\| parameter name="([A-Za-z_][A-Za-z0-9_]*)" type="(string|number|boolean|null|array|object)">')


def find_unescaped(text, token, start=0):
    pos = start
    while True:
        idx = text.find(token, pos)
        if idx < 0:
            return -1
        slash_count = 0
        j = idx - 1
        while j >= 0 and text[j] == "\\":
            slash_count += 1
            j -= 1
        if slash_count % 2 == 0:
            return idx
        pos = idx + 1


def find_json_parameter_end(text, token, start):
    in_string = False
    escaped = False
    i = start
    while i < len(text):
        char = text[i]
        if in_string:
            if escaped:
                escaped = False
            elif char == "\\":
                escaped = True
            elif char == '"':
                in_string = False
        else:
            if text.startswith(token, i):
                return i
            if char == '"':
                in_string = True
        i += 1
    return -1


def decode_byv_string(raw):
    out = []
    i = 0
    close_literal = "</|BYV| parameter>"
    while i < len(raw):
        if raw[i] != "\\":
            out.append(raw[i]); i += 1; continue
        if raw.startswith("\\\\", i):
            out.append("\\"); i += 2; continue
        if raw.startswith("\\n", i):
            out.append("\n"); i += 2; continue
        if raw.startswith("\\r", i):
            out.append("\r"); i += 2; continue
        if raw.startswith("\\t", i):
            out.append("\t"); i += 2; continue
        if raw.startswith("\\" + close_literal, i):
            out.append(close_literal); i += 1 + len(close_literal); continue
        raise ValueError("Invalid BYV string escape")
    return "".join(out)


def reject_json_constant(value):
    raise ValueError(f"Non-standard JSON constant is not allowed: {value}")


def reject_duplicate_json_keys(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"Duplicate JSON object key: {key}")
        result[key] = value
    return result


def parse_byv_calls(text):
    block_start = find_unescaped(text, CALLS_OPEN)
    if block_start < 0:
        return None
    if find_unescaped(text, "<|BYV|", 0) != block_start:
        raise ValueError("Unexpected BYV control token before calls block")
    pos = block_start + len(CALLS_OPEN)
    calls = []
    while True:
        while pos < len(text) and text[pos].isspace():
            pos += 1
        if text.startswith(CALLS_CLOSE, pos):
            block_end = pos
            pos += len(CALLS_CLOSE)
            break
        match = INVOKE_OPEN_RE.match(text, pos)
        if not match:
            raise ValueError("Invalid text or invoke opening token inside calls block")
        tool_name, call_id = match.groups()
        pos = match.end()
        args = {}
        while True:
            while pos < len(text) and text[pos].isspace():
                pos += 1
            if text.startswith(INVOKE_CLOSE, pos):
                pos += len(INVOKE_CLOSE)
                break
            p_match = PARAM_OPEN_RE.match(text, pos)
            if not p_match:
                raise ValueError("Invalid parameter element")
            param_name, value_type = p_match.groups()
            pos = p_match.end()
            if value_type == "string":
                param_end = find_unescaped(text, PARAM_CLOSE, pos)
            else:
                param_end = find_json_parameter_end(text, PARAM_CLOSE, pos)
            if param_end < 0:
                raise ValueError("Unclosed parameter")
            raw = text[pos:param_end]
            if param_name in args:
                raise ValueError("Duplicate parameter")
            if value_type == "string":
                value = decode_byv_string(raw)
            else:
                value = json.loads(
                    raw.strip(),
                    parse_constant=reject_json_constant,
                    object_pairs_hook=reject_duplicate_json_keys,
                )
                actual_type = (
                    "null" if value is None else "boolean" if isinstance(value, bool) else
                    "number" if isinstance(value, (int, float)) else "string" if isinstance(value, str) else
                    "array" if isinstance(value, list) else "object" if isinstance(value, dict) else "invalid"
                )
                if actual_type != value_type:
                    raise ValueError(f"Declared type {value_type}, body is {actual_type}")
                if value_type == "number" and isinstance(value, float) and not math.isfinite(value):
                    raise ValueError("Non-finite JSON number")
            args[param_name] = value
            pos = param_end + len(PARAM_CLOSE)
        calls.append({"id": call_id, "tool": tool_name, "arguments": args})
    if not calls:
        raise ValueError("Empty calls block")
    ids = [call["id"] for call in calls]
    if len(ids) != len(set(ids)):
        raise ValueError("Duplicate invocation IDs")
    if find_unescaped(text, CALLS_OPEN, pos) >= 0 or find_unescaped(text, CALLS_CLOSE, pos) >= 0:
        raise ValueError("Multiple or stray calls control tokens")
    if find_unescaped(text, "<|BYV|", pos) >= 0 or find_unescaped(text, "</|BYV|", pos) >= 0:
        raise ValueError("Unexpected BYV control token after calls block")
    prefix = text[:block_start]
    if find_unescaped(prefix, "<|BYV|", 0) >= 0 or find_unescaped(prefix, "</|BYV|", 0) >= 0:
        raise ValueError("Unexpected BYV control token before calls block")
    return calls


def classify_mode(text):
    start = find_unescaped(text, CALLS_OPEN)
    if start < 0:
        if find_unescaped(text, "<|BYV|", 0) >= 0 or find_unescaped(text, "</|BYV|", 0) >= 0:
            raise ValueError("Unexpected or malformed BYV control token in TEXT output")
        return "TEXT"
    calls = parse_byv_calls(text)
    close = find_unescaped(text, CALLS_CLOSE, start + len(CALLS_OPEN))
    if close < 0 or calls is None:
        raise ValueError("Incomplete BYV calls block")
    before = text[:start].strip()
    after = text[close + len(CALLS_CLOSE):].strip()
    return "MIXED" if before or after else "STRUCTURED"


## 3. Load base model and trained adapter

In [8]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required")
props = torch.cuda.get_device_properties(0)
GPU_NAME = props.name
GPU_VRAM_GB = round(props.total_memory / (1024**3), 2)
GPU_COMPUTE_CAPABILITY = torch.cuda.get_device_capability(0)
COMPUTE_DTYPE = torch.bfloat16 if GPU_COMPUTE_CAPABILITY[0] >= 8 and torch.cuda.is_bf16_supported() else torch.float16

TOKENIZER = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, use_fast=True)
if TOKENIZER.pad_token_id is None:
    TOKENIZER.pad_token = TOKENIZER.eos_token
TOKENIZER.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)
major_version = int(transformers.__version__.split(".")[0])
dtype_key = "dtype" if major_version >= 5 else "torch_dtype"
load_kwargs = {"revision": MODEL_REVISION, "quantization_config": bnb_config, "device_map":"auto", "low_cpu_mem_usage":True}
load_kwargs[dtype_key] = COMPUTE_DTYPE
try:
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
except TypeError:
    load_kwargs.pop("dtype", None)
    load_kwargs["torch_dtype"] = COMPUTE_DTYPE
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
model_commit = getattr(base_model.config, "_commit_hash", None)
model = PeftModel.from_pretrained(base_model, str(OUTPUT_DIR))
model.eval()
print("GPU:", GPU_NAME, "VRAM:", GPU_VRAM_GB, "GB")
print("Base model commit:", model_commit)
print("Adapter loaded:", OUTPUT_DIR)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

GPU: Tesla T4 VRAM: 14.56 GB
Base model commit: cdbee75f17c01a7cc42f958dc650907174af0554
Adapter loaded: /content/drive/MyDrive/BYV-AI/outputs/byv-ai-qwen3-4b-instruct-2507-byv-markup-inline-context-qlora-t4-v0.2


## 4. Generate on the test split and score format adherence

In [9]:
def generate_text(messages, max_new_tokens=MAX_NEW_TOKENS):
    prompt = TOKENIZER.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = TOKENIZER(prompt, return_tensors="pt").to(model.get_input_embeddings().weight.device)
    model.config.use_cache = True
    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=TOKENIZER.pad_token_id, eos_token_id=TOKENIZER.eos_token_id,
            use_cache=True,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    model.config.use_cache = False
    return TOKENIZER.decode(new_tokens, skip_special_tokens=True).strip()


def expected_calls(target):
    try:
        return parse_byv_calls(target)
    except Exception:
        return None

records = []
mode_correct = 0
text_nonempty = 0
byv_expected = 0
byv_parse_pass = 0
exact_call_match = 0
expected_tool_count = 0
matched_tool_count = 0
parameter_total = 0
parameter_matched = 0

for row in TEST_ROWS:
    messages = messages_with_inline_context(row)
    for assistant_index, target_msg in enumerate(messages):
        if target_msg["role"] != "assistant":
            continue
        prediction = generate_text(messages[:assistant_index])
        expected = target_msg["content"]
        expected_mode = classify_mode(expected)
        mode_parse_error = None
        try:
            predicted_mode = classify_mode(prediction)
        except Exception as exc:
            predicted_mode = "INVALID"
            mode_parse_error = str(exc)
        mode_ok = predicted_mode == expected_mode
        mode_correct += int(mode_ok)
        parsed_pred = None
        parse_error = None
        if expected_mode == "TEXT":
            text_nonempty += int(bool(prediction.strip()) and predicted_mode == "TEXT")
        else:
            byv_expected += 1
            try:
                parsed_pred = parse_byv_calls(prediction)
                if parsed_pred is None:
                    raise ValueError("Expected a BYV calls block, but the model returned no calls block")
                byv_parse_pass += 1
            except Exception as exc:
                parse_error = str(exc)
            expected_parsed = expected_calls(expected)
            if parsed_pred is not None and expected_parsed is not None:
                if parsed_pred == expected_parsed:
                    exact_call_match += 1
                for expected_call, actual_call in zip(expected_parsed, parsed_pred):
                    expected_tool_count += 1
                    matched_tool_count += int(expected_call["tool"] == actual_call["tool"])
                    for key, expected_value in expected_call["arguments"].items():
                        parameter_total += 1
                        parameter_matched += int(actual_call["arguments"].get(key, object()) == expected_value)
                # Count additional or missing calls as mismatches in exact-call rate.
            else:
                expected_parsed = expected_calls(expected)
                expected_tool_count += len(expected_parsed or [])
                parameter_total += sum(len(c["arguments"]) for c in (expected_parsed or []))
        records.append({
            "id": row["id"], "category": row.get("category"),
            "expected_mode": expected_mode, "predicted_mode": predicted_mode,
            "mode_correct": mode_ok, "byv_parse_valid": parsed_pred is not None if expected_mode != "TEXT" else None,
            "expected_calls": expected_calls(expected), "predicted_calls": parsed_pred,
            "parse_error": parse_error or mode_parse_error, "prediction": prediction,
        })

summary = {
    "base_model": MODEL_ID,
    "base_model_revision": MODEL_REVISION,
    "base_model_commit": model_commit,
    "dataset_version": DATASET_VERSION,
    "prompt_strategy": PROMPT_STRATEGY,
    "test_records": len(TEST_ROWS),
    "assistant_targets": len(records),
    "output_mode_accuracy": mode_correct / max(1, len(records)),
    "invalid_output_count": sum(1 for item in records if item["predicted_mode"] == "INVALID"),
    "text_nonempty_rate": text_nonempty / max(1, sum(1 for r in records if r["expected_mode"] == "TEXT")),
    "byv_expected_targets": byv_expected,
    "byv_parse_pass_rate": byv_parse_pass / max(1, byv_expected),
    "exact_tool_call_match_rate": exact_call_match / max(1, byv_expected),
    "tool_name_accuracy": matched_tool_count / max(1, expected_tool_count),
    "parameter_exact_match_rate": parameter_matched / max(1, parameter_total),
    "gpu_name": GPU_NAME,
    "gpu_vram_gb": GPU_VRAM_GB,
    "test_sha256": EXPECTED_TEST_SHA256,
    "semantic_quality_review": "manual review required; syntax metrics are not semantic accuracy",
}
(EXPERIMENT_DIR / "test_predictions.jsonl").write_text(
    "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in records), encoding="utf-8"
)
(EXPERIMENT_DIR / "evaluation_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(summary, ensure_ascii=False, indent=2))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{
  "base_model": "Qwen/Qwen3-4B-Instruct-2507",
  "base_model_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "base_model_commit": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "dataset_version": "byv-markup-v0.1",
  "prompt_strategy": "inline-byv-instructions-plus-per-record-tool-schema-v1",
  "test_records": 35,
  "assistant_targets": 39,
  "output_mode_accuracy": 0.8717948717948718,
  "invalid_output_count": 0,
  "text_nonempty_rate": 1.0,
  "byv_expected_targets": 10,
  "byv_parse_pass_rate": 1.0,
  "exact_tool_call_match_rate": 0.8,
  "tool_name_accuracy": 1.0,
  "parameter_exact_match_rate": 0.8461538461538461,
  "gpu_name": "Tesla T4",
  "gpu_vram_gb": 14.56,
  "test_sha256": "518fc6bdaee0cf8e48f2b070898a6f20e35bdff0d6b2202c6a4699fd2755b952",
  "semantic_quality_review": "manual review required; syntax metrics are not semantic accuracy"
}


## 5. Review checklist

- Periksa semua target yang salah mode.
- Periksa semua output BYV yang gagal diparse.
- Periksa tool name dan setiap parameter secara manual.
- Periksa kualitas jawaban TEXT secara semantik, faktual, dan bahasa.
- Jangan menganggap exact tool-call match sebagai bukti bahwa tool result atau jawaban akhirnya benar.

Artefak tersimpan di `MyDrive/BYV-AI/experiments/` pada folder run.